
# Navigating the Borel hierarchy lattice

This notebook drives **`borel_hierarchy_v1.py`**: a module that reads the
lattice diagram (`borel_hierarchy_lattice_v1.svg`, the id-annotated copy of
`borel_hierarchy_lattice.svg`) and builds a navigable graph of the Borel
classes.

The graph model:

- **nodes** are the classes — Σ⁰ₙ (countable unions, purple), Π⁰ₙ
  (countable intersections, teal), Δ⁰ₙ = Σ⁰ₙ ∩ Π⁰ₙ (gray), and the Borel
  sets at the top (the union of all levels below ω₁);
- an **edge** `u → v` means *u ⊆ v* (the arrow points upward);
- the two **dashed** edges (Σ⁰₃, Π⁰₃ → Borel sets) stand for the
  transfinite stretch through every countable ordinal.

What is new in v1 (code-review driven): a real **`walk()`** event stream,
**`Edge` objects** with metadata, stable **ids** on nodes and edges
(`node-sigma-2`, `edge-sigma-1-delta-2`), **hooks** (`on_enter` /
`on_traverse` / `dispatch`), a JSON **trace** for a browser player,
**`validate()`** against the taxonomy, and loud failures (parse warnings,
hard error on an empty graph).


In [1]:

from pathlib import Path
import sys

HERE = Path.cwd()
if not (HERE / "borel_hierarchy_v1.py").exists():
    sys.path.insert(0, str(HERE.parent))  # run from a subfolder?
from borel_hierarchy_v1 import load_graph, witness_grid

SVG = HERE / "borel_hierarchy_lattice_v1.svg"   # id-annotated diagram
g = load_graph(SVG)
print(repr(g))
print("parse warnings:", g.warnings or "none")


BorelGraph(classes=9, edges=10, source='D:\\Source\\hermes-dir\\borel_hierarchy_lattice_v1.svg')
parse warnings: none



## 1. The graph as data

Every class and edge has a stable id — the anchor point for hooks and for
the animation player.  `data-*` attributes from the SVG (family, level,
the expansion `prompt`) are read back onto the node.


In [2]:

import pandas as pd

classes = pd.DataFrame([
    {"name": n.name, "id": n.id, "subtitle": n.subtitle,
     "family": n.family, "level": n.level,
     "contained_in": ", ".join(g.successors(n.name)) or "— (top)"}
    for n in sorted(g.nodes.values(), key=lambda n: (n.rank, n.x))
])
edges = pd.DataFrame([
    {"id": e.id, "from": e.src, "to": e.dst, "means": f"{e.src} ⊆ {e.dst}",
     "dashed": e.dashed, "metadata": e.metadata or {}}
    for e in sorted(g.edges.values(), key=lambda e: e.id)
])
classes


,name,id,subtitle,family,level,contained_in
0,Σ⁰₁,node-sigma-1,open sets,sigma,1,Δ⁰₂
1,Π⁰₁,node-pi-1,closed sets,pi,1,Δ⁰₂
2,Σ⁰₂,node-sigma-2,F_σ sets,sigma,2,Δ⁰₃
3,Δ⁰₂,node-delta-2,,delta,2,"Σ⁰₂, Π⁰₂"
4,Π⁰₂,node-pi-2,G_δ sets,pi,2,Δ⁰₃
5,Σ⁰₃,node-sigma-3,G_δσ sets,sigma,3,Borel sets
6,Δ⁰₃,node-delta-3,,delta,3,"Σ⁰₃, Π⁰₃"
7,Π⁰₃,node-pi-3,F_σδ sets,pi,3,Borel sets
8,Borel sets,node-borel,union of all levels α < ω₁,borel,omega1,— (top)


In [3]:

edges


,id,from,to,means,dashed,metadata
0,edge-delta-2-pi-2,Δ⁰₂,Π⁰₂,Δ⁰₂ ⊆ Π⁰₂,False,{}
1,edge-delta-2-sigma-2,Δ⁰₂,Σ⁰₂,Δ⁰₂ ⊆ Σ⁰₂,False,{}
2,edge-delta-3-pi-3,Δ⁰₃,Π⁰₃,Δ⁰₃ ⊆ Π⁰₃,False,{}
3,edge-delta-3-sigma-3,Δ⁰₃,Σ⁰₃,Δ⁰₃ ⊆ Σ⁰₃,False,{}
4,edge-pi-1-delta-2,Π⁰₁,Δ⁰₂,Π⁰₁ ⊆ Δ⁰₂,False,{}
5,edge-pi-2-delta-3,Π⁰₂,Δ⁰₃,Π⁰₂ ⊆ Δ⁰₃,False,{}
6,edge-pi-3-borel,Π⁰₃,Borel sets,Π⁰₃ ⊆ Borel sets,True,{'dashed': 'true'}
7,edge-sigma-1-delta-2,Σ⁰₁,Δ⁰₂,Σ⁰₁ ⊆ Δ⁰₂,False,{}
8,edge-sigma-2-delta-3,Σ⁰₂,Δ⁰₃,Σ⁰₂ ⊆ Δ⁰₃,False,{}
9,edge-sigma-3-borel,Σ⁰₃,Borel sets,Σ⁰₃ ⊆ Borel sets,True,{'dashed': 'true'}


In [4]:

# data-* attributes read back from the annotated SVG
node = g.node("Σ⁰₂")
print(node.id, "|", node.subtitle, "|", node.family, "level", node.level)
print("prompt:", node.data.get("prompt"))


node-sigma-2 | F_σ sets | sigma level 2
prompt: Explain F_σ sets: countable unions of closed sets. Example: ℚ ∩ [0,1], a countable union of points.



## 2. Queries (v0, unchanged)

`ancestors` / `descendants` are transitive; `contains` answers
*u ⊆ v?*; `shortest_path` gives an inclusion chain; `complement` and
`build` give the semantics.


In [5]:

print("ancestors(open)   =", sorted(g.ancestors("open")))
print("descendants(Δ⁰₂)  =", sorted(g.descendants("Δ⁰₂")))
print()
print("Π⁰₂ contains Σ⁰₁ (every open set is G_δ)?", g.contains("Π⁰₂", "Σ⁰₁"))
print("Π⁰₂ contains Σ⁰₂ (F_σ ⊆ G_δ)?           ", g.contains("Π⁰₂", "Σ⁰₂"))
print()
print("shortest_path(open → G_δ):", " → ".join(g.shortest_path("open", "G_δ")))
print("shortest_path(Π⁰₁ → Borel):", " → ".join(g.shortest_path("Π⁰₁", "Borel sets")))
print()
print("complement(F_σ) =", g.complement("F_σ"))
print("build(Σ⁰₂)      =", g.build("Σ⁰₂"))
print("build(Π⁰₃)      =", g.build("Π⁰₃"))


ancestors(open)   = ['Borel sets', 'Δ⁰₂', 'Δ⁰₃', 'Π⁰₂', 'Π⁰₃', 'Σ⁰₂', 'Σ⁰₃']
descendants(Δ⁰₂)  = ['Π⁰₁', 'Σ⁰₁']

Π⁰₂ contains Σ⁰₁ (every open set is G_δ)? True
Π⁰₂ contains Σ⁰₂ (F_σ ⊆ G_δ)?            False

shortest_path(open → G_δ): Σ⁰₁ → Δ⁰₂ → Π⁰₂
shortest_path(Π⁰₁ → Borel): Π⁰₁ → Δ⁰₂ → Σ⁰₂ → Δ⁰₃ → Σ⁰₃ → Borel sets

complement(F_σ) = Π⁰₂
build(Σ⁰₂)      = countable unions of Π⁰₁ sets
build(Π⁰₃)      = countable intersections of Σ⁰₂ sets



## 3. Traversal: `walk()`

`walk(start, direction)` is a **generator of events**:

- `("node", name)` — the walk is *at* a class;
- `("edge", Edge)` — the walk *crosses* an arrow (every outgoing edge of
  the current node is crossed, even when the far end was already seen).

Breadth-first, deterministic, in the visual reading order of the diagram.
`direction="up"` climbs toward the Borel sets; `"down"` descends.


In [6]:

events = list(g.walk("open"))
for kind, item in events:
    if kind == "node":
        print(f"  enter  {item}")
    else:
        mark = "  (dashed)" if item.dashed else ""
        print(f"  cross  {item.src} -> {item.dst}{mark}")


  enter  Σ⁰₁
  cross  Σ⁰₁ -> Δ⁰₂
  enter  Δ⁰₂
  cross  Δ⁰₂ -> Σ⁰₂
  cross  Δ⁰₂ -> Π⁰₂
  enter  Σ⁰₂
  cross  Σ⁰₂ -> Δ⁰₃
  enter  Π⁰₂
  cross  Π⁰₂ -> Δ⁰₃
  enter  Δ⁰₃
  cross  Δ⁰₃ -> Σ⁰₃
  cross  Δ⁰₃ -> Π⁰₃
  enter  Σ⁰₃
  cross  Σ⁰₃ -> Borel sets  (dashed)
  enter  Π⁰₃
  cross  Π⁰₃ -> Borel sets  (dashed)
  enter  Borel sets


In [7]:

# descending from the top: the same lattice, walked in reverse
down = [(kind, item) for kind, item in g.walk("Borel sets", direction="down")]
print(" → ".join(name for kind, name in down if kind == "node"))


Borel sets → Σ⁰₃ → Π⁰₃ → Δ⁰₃ → Σ⁰₂ → Π⁰₂ → Δ⁰₂ → Σ⁰₁ → Π⁰₁



## 4. Hooks: activities attached to nodes and edges

Hooks are the "further process" the review asked for.  They key on the
stable ids, so they survive re-parsing and map 1:1 onto the SVG elements:

- `on_enter(name, fn)` — `fn(node_name)` when the walk enters a class;
- `on_traverse(match, fn)` — `fn(edge)` when the walk crosses a matching
  edge.  `match` is an edge id, a `(src, dst)` pair, or a predicate
  (e.g. `lambda e: e.dashed`).

A hook may return a dict: `run_walk` merges it into the payload of the
trace step, so an activity can *annotate* the walk for the browser player.


In [8]:

g2 = load_graph(SVG)
log = []

# activity on a node: announce the canonical example when the walk arrives
def enter_sigma2(name):
    log.append(f"enter {name}: show example")
    return {"activity": "show_example", "example": "ℚ ∩ [0,1]"}

# activity on a vertex/edge: pulse every arrow as it is crossed
def cross_all(e):
    log.append(f"cross {e.id}")
    return None

# activity on specific edges: flag the transfinite stretch
def cross_dashed(e):
    log.append(f"cross {e.id}  -> transfinite stretch (levels continue to ω₁)")
    return {"activity": "flag_transfinite"}

_ = g2.on_enter("Σ⁰₂", enter_sigma2)
_ = g2.on_enter("Π⁰₃", lambda n: log.append(f"enter {n}: convergence set ⋂ₖ⋃_N⋂{{n,m≥N}}"))
_ = g2.on_traverse(cross_all)
_ = g2.on_traverse(lambda e: e.dashed, cross_dashed)

steps = g2.run_walk("open")          # walk + dispatch + record
for line in log:
    print(line)


cross edge-sigma-1-delta-2
cross edge-delta-2-sigma-2
cross edge-delta-2-pi-2
enter Σ⁰₂: show example
cross edge-sigma-2-delta-3
cross edge-pi-2-delta-3
cross edge-delta-3-sigma-3
cross edge-delta-3-pi-3
cross edge-sigma-3-borel
cross edge-sigma-3-borel  -> transfinite stretch (levels continue to ω₁)
enter Π⁰₃: convergence set ⋂ₖ⋃_N⋂{n,m≥N}
cross edge-pi-3-borel
cross edge-pi-3-borel  -> transfinite stretch (levels continue to ω₁)


In [9]:

# dispatch() can fire hooks on a single event, standalone
g2.dispatch("node", "Σ⁰₂")
g2.dispatch("edge", "edge-pi-3-borel")
print("after manual dispatch:", log[-2:])
g2.clear_hooks()


after manual dispatch: ['cross edge-pi-3-borel', 'cross edge-pi-3-borel  -> transfinite stretch (levels continue to ω₁)']



## 5. The trace: JSON for the browser player

"Trace, then replay" (Answer 3): Python records the walk as a JSON list of
steps — the mathematics stays in one testable place, a small JS player
animates the steps on the SVG (targeting the `id` attributes):

```json
[{"enter": "Σ⁰₁", "payload": {...}},
 {"cross": ["Σ⁰₁", "Δ⁰₂"], "edge": "edge-sigma-1-delta-2",
  "dashed": false, "payload": {}}, ...]
```

`enter` payloads carry level, family, the build recipe and the canonical
examples; `cross` payloads carry the edge metadata **plus whatever the
hooks returned** (the `activity` annotations from §4).


In [10]:

import json

g3 = load_graph(SVG)
_ = g3.on_enter("Σ⁰₂", lambda n: {"activity": "show_example", "example": "ℚ ∩ [0,1]"})
_ = g3.on_traverse(lambda e: e.dashed, lambda e: {"activity": "flag_transfinite"})
steps = g3.run_walk("open")

TRACE = HERE / "Borel-Hierarchy-trace_v1.json"
TRACE.write_text(g3.to_trace_json(steps), encoding="utf-8")
print(f"wrote {TRACE.name} with {len(steps)} steps")
print()
print(json.dumps(steps[:6], ensure_ascii=False, indent=2))


wrote Borel-Hierarchy-trace_v1.json with 17 steps

[
  {
    "enter": "Σ⁰₁",
    "payload": {
      "level": 1,
      "family": "sigma",
      "build": "open sets",
      "examples": [
        "any open set, e.g. (0, 1)"
      ]
    }
  },
  {
    "cross": [
      "Σ⁰₁",
      "Δ⁰₂"
    ],
    "edge": "edge-sigma-1-delta-2",
    "dashed": false,
    "payload": {}
  },
  {
    "enter": "Δ⁰₂",
    "payload": {
      "level": 2,
      "family": "delta",
      "build": "the sets that are both Σ⁰₂ and Π⁰₂",
      "examples": []
    }
  },
  {
    "cross": [
      "Δ⁰₂",
      "Σ⁰₂"
    ],
    "edge": "edge-delta-2-sigma-2",
    "dashed": false,
    "payload": {}
  },
  {
    "cross": [
      "Δ⁰₂",
      "Π⁰₂"
    ],
    "edge": "edge-delta-2-pi-2",
    "dashed": false,
    "payload": {}
  },
  {
    "enter": "Σ⁰₂",
    "payload": {
      "level": 2,
      "family": "sigma",
      "build": "countable unions of Π⁰₁ sets",
      "examples": [
        "ℚ ∩ [0,1]: a countable union of points (s

In [11]:

# the hook-annotated steps, mid-walk
for s in steps:
    act = s.get("payload", {}).get("activity")
    if act:
        target = s.get("enter") or " -> ".join(s["cross"])
        print(f"{s.get('enter') or 'cross ' + ' -> '.join(s['cross']):28s} activity: {act}")


Σ⁰₂                          activity: show_example
cross Σ⁰₃ -> Borel sets      activity: flag_transfinite
cross Π⁰₃ -> Borel sets      activity: flag_transfinite



## 6. The witness grid (depth-limited verdict)

The presentation's teaching device, evaluated by the module.  **The grid
is a finite truncation**: "some row is all green" means all green *across
the columns shown* (and, with `depth`, only the first `depth` rows).  It
is a stage of the limit — a row green "forever" can never be certified by
any finite grid, so a lesson must show the depth, not a settled verdict.


In [12]:

# F_σ (Σ⁰₂):  x ∈ ⋃ₙ ⋂ₘ Aₙ,ₘ  <=>  some row is all green
grid_fs = [[True,  True,  True,  True],
           [False, True,  False, True],
           [True,  False, True,  False]]
for depth in (1, 2, 3):
    print(f"depth {depth}: exists-all =",
          witness_grid(grid_fs, "exists-all", depth=depth))

print()
# G_δ (Π⁰₂):  x ∈ ⋂ₙ ⋃ₘ Aₙ,ₘ  <=>  green in every row
grid_gd = [[False, False, True],
           [True,  False, False],
           [False, False, False]]
for depth in (1, 2, 3):
    print(f"depth {depth}: all-exists =",
          witness_grid(grid_gd, "all-exists", depth=depth))
print("(at depth 3 the third row has no green cell -> not a member —")
print(" at depth 2 the verdict was still 'member': the limit is not settled)")


depth 1: exists-all = True
depth 2: exists-all = True
depth 3: exists-all = True

depth 1: all-exists = True
depth 2: all-exists = True
depth 3: all-exists = False
(at depth 3 the third row has no green cell -> not a member —
 at depth 2 the verdict was still 'member': the limit is not settled)



## 7. `validate()`: the taxonomy is the source of truth

The structure follows entirely from the taxonomy (Σ, Π, Δ at each level);
the SVG only supplies geometry.  `validate()` cross-checks the parsed
edges against the taxonomy-derived expectation:

- per level *n*: Σ⁰ₙ → Δ⁰ₙ₊₁ and Π⁰ₙ → Δ⁰ₙ₊₁ (climbing), plus
  Δ⁰ₙ → Σ⁰ₙ and Δ⁰ₙ → Π⁰ₙ (same level — Δ sits inside both);
- the top level's Σ⁰ₙ/Π⁰ₙ point up to the Borel sets (dashed).


In [13]:

v = g.validate()
print("validate():", "OK — matches the taxonomy" if v["ok"] else "MISMATCH")
print("missing:", v["missing"] or "none", "| extra:", v["extra"] or "none")

# tamper with the graph -> validate() catches it
g_bad = load_graph(SVG)
del g_bad.edges["edge-delta-2-sigma-2"]
g_bad._build_adjacency()
print()
print("after deleting edge Δ⁰₂ -> Σ⁰₂:")
print("  validate():", "OK" if g_bad.validate()["ok"] else "MISMATCH",
      "| missing:", g_bad.validate()["missing"])


validate(): OK — matches the taxonomy
missing: none | extra: none

after deleting edge Δ⁰₂ -> Σ⁰₂:
  validate(): MISMATCH | missing: [('Δ⁰₂', 'Σ⁰₂')]


In [14]:

# loud failures: an SVG with no known boxes raises instead of returning
# a silent empty graph
try:
    from borel_hierarchy_v1 import BorelGraph
    BorelGraph.from_svg(HERE / "Borel-Sets-Presentation_2.md")
except ValueError as exc:
    print("ValueError:", exc)


ValueError: could not parse D:\Source\hermes-dir\Borel-Sets-Presentation_2.md: not well-formed (invalid token): line 1, column 1



## 8. The id-annotated SVG

`borel_hierarchy_lattice_v1.svg` is the diagram with the anchors the
player needs: each box group carries `id="node-<family>-<level>"` plus
`data-family`, `data-level`, `data-prompt` (the expansion prompt asked
when a box is clicked); each arrow carries `id="edge-<srcslug>-<dstslug>"`
and `data-dashed="true"` for the transfinite ones.  Regenerate it with:

```bash
python borel_hierarchy_v1.py --annotate
```

(then rename the output to `borel_hierarchy_lattice_v1.svg`).


In [15]:

import re
raw = SVG.read_text(encoding="utf-8")
node_ids = re.findall(r'id="(node-[^"]+)"', raw)
edge_ids = re.findall(r'id="(edge-[^"]+)"', raw)
print("node ids:", len(node_ids))
for i in node_ids:
    print("  ", i)
print("edge ids:", len(edge_ids))
for i in edge_ids:
    print("  ", i)


node ids: 9
   node-borel
   node-sigma-3
   node-pi-3
   node-delta-3
   node-sigma-2
   node-pi-2
   node-delta-2
   node-sigma-1
   node-pi-1
edge ids: 10
   edge-sigma-3-borel
   edge-pi-3-borel
   edge-sigma-1-delta-2
   edge-pi-1-delta-2
   edge-delta-2-sigma-2
   edge-delta-2-pi-2
   edge-sigma-2-delta-3
   edge-pi-2-delta-3
   edge-delta-3-sigma-3
   edge-delta-3-pi-3



## Recap

| need (from the review) | v1 answer |
| --- | --- |
| traversal, not just queries | `walk()` yields `("node", name)` / `("edge", Edge)` in order |
| edge objects to hang callbacks on | `Edge` dataclass (id, src, dst, dashed, metadata) in `_out`/`_in` |
| stable ids for hooks | `node-sigma-2`, `edge-sigma-1-delta-2`, … |
| parser keeps presentation data | `id` + `data-*` read back onto nodes/edges; `annotate_svg()` writes the annotated diagram |
| no silent failures | `g.warnings` list; `ValueError` on non-XML or zero classes |
| corrected statements | docstring no longer claims strict n→n+1 grading; no `Σ¹₀` alias; `witness_grid` is declared depth-limited; `validate()` cross-checks the SVG against the taxonomy |
| trace for the browser player | `run_walk()` → JSON steps with hook-merged payloads (`Borel-Hierarchy-trace_v1.json`) |
